In [16]:
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from pinecone import Pinecone
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore

In [4]:
load_dotenv()

True

In [5]:
embedding_function = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [8]:
len(embedding_function.embed_query("Hello world"))


768

In [6]:
import os
pinecone_api_key=os.getenv("PINECONE_API_KEY")

In [7]:
pc=Pinecone(api_key=pinecone_api_key)

In [9]:
from pinecone import ServerlessSpec
index_name="book-recommendations"   
pc.has_index(index_name)

True

In [14]:
if not pc.has_index(index_name):
    pc.create_index(
    name=index_name,
    dimension=768,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws",region="us-east-1")    
)

In [10]:
index=pc.Index(index_name)

In [25]:
from langchain_pinecone import PineconeVectorStore
vector_store=PineconeVectorStore(index=index,embedding=embedding_function)
results = vector_store.similarity_search("I want to read about Christie for Christmas")


In [11]:
loader = TextLoader(
        "E:/book_recommender-main/book_recommender-main/book_recommender/data/tagged_description.txt",
        encoding='utf-8'
    )

In [12]:
raw_documents=loader.load()

In [13]:
len(raw_documents)

1

In [17]:
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=0, chunk_overlap=0)

In [18]:
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1168, which is longer than the specified 0
Created a chunk of size 1214, which is longer than the specified 0
Created a chunk of size 373, which is longer than the specified 0
Created a chunk of size 309, which is longer than the specified 0
Created a chunk of size 483, which is longer than the specified 0
Created a chunk of size 482, which is longer than the specified 0
Created a chunk of size 960, which is longer than the specified 0
Created a chunk of size 188, which is longer than the specified 0
Created a chunk of size 843, which is longer than the specified 0
Created a chunk of size 296, which is longer than the specified 0
Created a chunk of size 197, which is longer than the specified 0
Created a chunk of size 881, which is longer than the specified 0
Created a chunk of size 1088, which is longer than the specified 0
Created a chunk of size 1189, which is longer than the specified 0
Created a chunk of size 304, which is longer than the specified 0
Create

In [19]:
len(documents)

5088

In [20]:
from uuid import uuid4
uuids = [str(uuid4()) for _ in range(len(documents))]

In [21]:
len(uuids)

5088

In [30]:
documents[0]


Document(metadata={'source': 'E:/book_recommender-main/book_recommender-main/book_recommender/data/tagged_description.txt', 'text': '9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds

In [26]:
vector_store.add_documents(documents=documents, ids=uuids)

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Fri, 13 Jun 2025 08:59:43 GMT', 'Content-Type': 'application/json', 'Content-Length': '118', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '70633', 'x-pinecone-request-id': '1184577577888757527', 'x-envoy-upstream-service-time': '2', 'server': 'envoy'})
HTTP response body: {"code":11,"message":"Error, message length too large: found 4584440 bytes, the limit is: 4194304 bytes","details":[]}


### Here we have a limitation with the pinecone. So we moved with the astradb.

In [36]:
import sys
import tqdm
for i in tqdm.tqdm(range(len(documents))):
    if sys.getsizeof(documents[i])>100:
        print(i)

100%|██████████| 5088/5088 [00:00<00:00, 727008.88it/s]


In [38]:
pip install -qU langchain-astradb


Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.


In [39]:
from langchain_astradb import AstraDBVectorStore

In [44]:
api_endpoint = os.getenv('ASTRA_API_ENDPOINT')
TOKEN = os.getenv('ASTRA_DB_TOKEN')

In [ ]:
vector_store = AstraDBVectorStore(
    embedding=embedding_function,
    api_endpoint=api_endpoint,
    collection_name="astra_vector_langchain",
    token=TOKEN,
    namespace='book_recommendation')

In [42]:
vector_store.add_documents(documents=documents)

['f08694b61d374bd3b5811271034af549',
 '39b35072c1e44cac9b3bcb6c366168e6',
 '55f61b19e0ac41b397ce7369db79e67a',
 '9c4e60ea4fb5420aaafddc594ec44b09',
 '65ec9e2cb3d243b9894cbe0a46e4b326',
 'c11d3a1dbf45442c84be486af77e749d',
 'c4fa086f300b414689a74ec475ba6cef',
 'c521bf2d946b4cb3b32ab66e4dcbd983',
 'aa8ced78e5d04cc18a5d1a258f40217c',
 'a001d6d1cc3f4d92b822f7ad91eec62d',
 '59dcb63672db4f91865f179d94d16167',
 '7803359dc48848698e9fe646803b9918',
 '04544795279d43dd984adfbfeda5cedb',
 '1e7db6e40a654184949d369186f05b43',
 'ece5e86a54154a069b4c1de40afda1e5',
 'e888b3808b16487d9eec332286cf9ca0',
 '95239c687625434b9403909a26e65cad',
 '4901ab91f6034792974310ab94630e12',
 'f9653fa7e264442997c29915ba686c85',
 'b302ed81aded4addb7cf7b0afb7d4932',
 'bb670e759b87461c9a76cb53f32e6b8a',
 'c49ff604e3b642ef96c52b160c7ccc0e',
 'be401aa473ac43d1a4fdf15190969e96',
 '39a31163a8c34aeebfa37c9c4b4eb59d',
 'ac3af10ab6b9499ba387f47a2aa5361c',
 '0751c0f4cdd5435291346a4820cca2ce',
 '640963f5686f4cf797c8f5f687a8f4fc',
 

In [43]:
vector_store.similarity_search("I want to read about Christie for Christmas")

[Document(id='1e7db6e40a654184949d369186f05b43', metadata={'source': 'E:/book_recommender-main/book_recommender-main/book_recommender/data/tagged_description.txt', 'text': '9780006490456 Newly-Jacketed Edition Designed To Celebrate The 50Th Anniversary Of Christie S Faultlessly Plotted Witness For The Prosecution And Other Outstanding Plays. The Perfect Complement To The Latest Edition Of The Mousetrap And Selected Plays (50Th Aniversary Edition). Headlining This Book Is Witness For The Prosecution Christie S Highly Successful Stage Play Which Won The New York Drama Critics Circle Award For Best Foreign Play. A Stunning Courtroom Drama, It Tells The Story Of A Scheming Wife Testifying Against Her Husband In A Shocking Murder Trial. The Wild Beauty Of A Seaside House Perched High On The Devonshire River Tern Provides A Stunning Back-Drop In Towards Zero As A Psychopathic Murderer Homes In On The Unsuspecting Victims. Passion, Murder And Love Are The Deadly Ingredients In Verdict, Making